# 02 — Entrenar VolleyDetector ③ (custom, sin librería YOLO)

Entrena nuestra red `VolleyDetector` (definida en `src/models/detector.py`) con el dataset
`voley.yolov8`. T4 free de Colab tarda ~30–45 min con 80 épocas.

Output: `weights/detector.pt` (descarga al final manualmente al PC para la GUI).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf volley-ai
!git clone https://github.com/Angelote567/DeepLearning.git volley-ai
%cd volley-ai
!pip install -q torch torchvision tqdm Pillow PyYAML
import os
os.makedirs('data', exist_ok=True)
# Symlinks a Drive (dataset descargado por notebook 01, weights persistentes)
if not os.path.exists('data/volleyball-2'):
    os.symlink('/content/drive/MyDrive/volley-ai/data/volleyball-2', 'data/volleyball-2')
if os.path.exists('weights') and not os.path.islink('weights'):
    if not os.listdir('weights') or os.listdir('weights') == ['.gitkeep']:
        import shutil; shutil.rmtree('weights')
if not os.path.exists('weights'):
    os.symlink('/content/drive/MyDrive/volley-ai/weights', 'weights')

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
from src.train.train_detector import train
best = train(epochs=80, batch_size=16, lr=1e-3)
print('Mejores pesos:', best)

In [ ]:
# Evaluación rápida con imágenes del split valid (no siempre hay test/)
import torch
from PIL import Image
import torchvision.transforms.functional as TF
from src.models.detector import VolleyDetector
from src.config import IMG_SIZE, CLASSES

model = VolleyDetector().cuda().eval()
ckpt = torch.load('weights/detector.pt', map_location='cuda')
model.load_state_dict(ckpt['model'])

import glob, random, os
test_dir = 'data/volleyball-2/test/images' if os.path.exists('data/volleyball-2/test/images') else 'data/volleyball-2/valid/images'
for path in random.sample(glob.glob(f'{test_dir}/*'), 4):
    img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    x = TF.to_tensor(img).unsqueeze(0).cuda()
    out = model.predict(x)[0]
    print(os.path.basename(path), '->', len(out['boxes']), 'detecciones',
          [CLASSES[int(c)] for c in out['classes'].tolist()])